# Lecture 7 examples: Finishing SMM + Parametric Bootstrap

Code behind today's slides.

In [ ]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # set once, here -- never inside a simulator function

# Sampling distribution of the coin-flip estimator

Estimate the probability `p` that a coin lands heads based on ten coin flips, using the proportion of heads as the estimator, `p_hat = h(Y) = number of heads / 10`.

## The sampling distribution

Suppose the true value is `p = 0.6`. Simulate the entire experiment (flip the coin 10 times, calculate `p_hat`, save it) many times.

In [ ]:
p_true = 0.6
n1 = 10
B = 3000

rng_c1 = np.random.default_rng(101)
phat_n10 = rng_c1.binomial(n1, p_true, size=B) / n1

fig, ax = plt.subplots()
ax.hist(phat_n10, bins=np.arange(-0.05, 1.15, 1 / n1), color="steelblue", edgecolor="white")
ax.axvline(p_true, color="firebrick", linestyle="--")
ax.set_xlabel(r"$\hat p$"); ax.set_ylabel("count")
ax.set_title("Sampling distribution of p-hat, n = 10, 3000 experiments")
plt.show()

This histogram contains 3,000 estimates, not 3,000 individual coin flips.

## A different experiment, a different sampling distribution

For a dataset of `n` coin flips, the estimator is `p_hat_n = mean(Y_1,...,Y_n)`. Changing `n` gives a different estimator, with its own sampling distribution.

In [ ]:
sizes = [10, 50, 200]
rng_c2 = np.random.default_rng(102)
by_size = {n: rng_c2.binomial(n, p_true, size=B) / n for n in sizes}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, n in zip(axes, sizes):
    ax.hist(by_size[n], bins=40, color="steelblue", edgecolor="white")
    ax.axvline(p_true, color="firebrick", linestyle="--")
    ax.set_title(f"n = {n}")
    ax.set_xlabel(r"$\hat p$")
axes[0].set_ylabel("count")
fig.suptitle("Sampling distribution of p-hat at three sample sizes (true p = 0.6)")
plt.show()

Experiments with more data produce sampling distributions that are more tightly concentrated around the true value.

# Bootstrapping the coin flip

With real data, $\theta_0$ is unknown; the parametric bootstrap replaces it with the fitted $\hat\theta$. Take the dataset `H H H H T H H T H H` (8 heads out of 10, so `p_hat = 0.8`) as the observed data, and follow the bootstrap algorithm:

1. Fit: `p_hat = h(Y_obs) = 0.8`.
2. Simulate `B = 3000` new datasets of 10 flips from `Bernoulli(0.8)`.
3. Refit `p_hat*(b)` on each.
4. Look at the distribution of the `p_hat*(b)`'s.

In [ ]:
dataset3 = np.array([1, 1, 1, 1, 0, 1, 1, 0, 1, 1])
phat_obs = dataset3.mean()
phat_obs

In [ ]:
rng_c3 = np.random.default_rng(103)
phat_boot = rng_c3.binomial(n1, phat_obs, size=B) / n1

se_boot = phat_boot.std(ddof=1)
ci_boot = np.quantile(phat_boot, [0.025, 0.975])
print("estimate", phat_obs, "se", se_boot, "ci", ci_boot)

In [ ]:
fig, ax = plt.subplots()
ax.hist(phat_boot, bins=np.arange(-0.05, 1.15, 1 / n1), color="steelblue", edgecolor="white")
ax.axvline(phat_obs, color="firebrick", linestyle="--")
ax.set_xlabel(r"$\hat p^{*}$"); ax.set_ylabel("count")
ax.set_title("Bootstrap distribution of p-hat, repeat at the fitted p-hat = 0.8")
plt.show()

# Full simulation-based SMM: the Normal location-scale model

`Y_i ~ N(mu, sigma^2)`, `theta = (mu, sigma)`, fit using the identifiable summary pair `s(Y) = (Ybar, q_.75(Y))`. SMM is not the sensible way to fit a Normal model; we use it because we understand it well enough to check whether the bootstrap is doing what we intend. True values (known here only for pedagogical reasons): `mu0 = 5`, `sigma0 = 2`, `n = 40`.

## The estimator we will use

For a candidate `theta = (mu, sigma)`, simulate `R` datasets of size `n = 40` and estimate the expected summaries, `m_hat_R(theta) = mean over r of s(Y_theta^(r))`. Then minimize `Q_Y(theta) = [s(Y) - m_hat_R(theta)]' W [s(Y) - m_hat_R(theta)]`. The estimator is the complete numerical procedure `h(Y) = argmin_(mu,sigma) Q_Y(mu, sigma)`, found with a numerical optimizer -- not the model's closed-form solution.

In [ ]:
mu0, sigma0, n = 5.0, 2.0, 40
R_inner, B_outer = 2000, 3000
W = np.eye(2)

rng_z = np.random.default_rng(2024)
Z_bank = rng_z.normal(size=(R_inner, n))
rowMeanZ = Z_bank.mean(axis=1)

def make_mhat(p):
    rowQZ = np.quantile(Z_bank, p, axis=1)
    mbar = rowMeanZ.mean()
    qbar = rowQZ.mean()
    # Y_theta^(r) = mu + sigma * Z^(r), simulated once via Z_bank and
    # reused for every candidate theta the optimizer tries.
    def mhat(mu, sigma):
        return np.array([mu + sigma * mbar, mu + sigma * qbar])
    return mhat

def fit_smm_full(y, p, mhat_fun, W=np.eye(2), start=None):
    s_obs = np.array([y.mean(), np.quantile(y, p)])
    def Qfun(theta):
        g = s_obs - mhat_fun(theta[0], theta[1])
        return float(g @ W @ g)
    if start is None:
        start = [y.mean(), max(y.std(ddof=1), 0.1)]
    res = minimize(Qfun, start, method="L-BFGS-B",
                    bounds=[(None, None), (0.05, None)])
    return res.x

mhat_75 = make_mhat(0.75)

## Step 0: The oracle sampling distribution (not part of the bootstrap)

Only possible because this is a teaching simulation and we know `theta0 = (mu0, sigma0) = (5, 2)`. Repeat the complete experiment many times: simulate a dataset at `theta0`, fit it, record the estimate.

In [ ]:
rng_o = np.random.default_rng(41)
oracle_fits = np.array([
    fit_smm_full(rng_o.normal(mu0, sigma0, size=n), 0.75, mhat_75, W)
    for _ in range(B_outer)
])

## Step 1: Fit one observed dataset

In [ ]:
rng_obs = np.random.default_rng(42)
y_observed = rng_obs.normal(mu0, sigma0, size=n)
theta_hat = fit_smm_full(y_observed, 0.75, mhat_75, W)
theta_hat

## Step 2: Construct the bootstrap distribution

Simulate new datasets from the fitted model, refit each with exactly the same procedure, and record the estimate.

In [ ]:
rng_b = np.random.default_rng(43)
boot_fits = np.array([
    fit_smm_full(rng_b.normal(theta_hat[0], theta_hat[1], size=n), 0.75, mhat_75, W)
    for _ in range(B_outer)
])

## Comparing the oracle and bootstrap distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
axes[0].scatter(oracle_fits[:, 0], oracle_fits[:, 1], alpha=0.12, s=8, color="steelblue")
axes[0].set_title("Oracle (repeat at true theta0)")
axes[1].scatter(boot_fits[:, 0], boot_fits[:, 1], alpha=0.12, s=8, color="steelblue")
axes[1].set_title("Bootstrap (repeat at fitted theta-hat)")
for ax in axes:
    ax.set_xlabel(r"$\hat\mu$")
axes[0].set_ylabel(r"$\hat\sigma$")
fig.suptitle("Two sampling distributions of the same SMM estimator")
plt.show()

## What we calculate from the bootstrap fits

In [ ]:
se_oracle = oracle_fits.std(axis=0, ddof=1)
bias_oracle = oracle_fits.mean(axis=0) - np.array([mu0, sigma0])
se_boot = boot_fits.std(axis=0, ddof=1)
bias_boot = boot_fits.mean(axis=0) - theta_hat

print("quantity        oracle check   bootstrap estimate")
print(f"SE(mu)          {se_oracle[0]:.3f}          {se_boot[0]:.3f}")
print(f"SE(sigma)       {se_oracle[1]:.3f}          {se_boot[1]:.3f}")
print(f"bias(mu)        {bias_oracle[0]:.3f}         {bias_boot[0]:.3f}")
print(f"bias(sigma)     {bias_oracle[1]:.3f}         {bias_boot[1]:.3f}")

print("95% CI mu:", np.quantile(boot_fits[:, 0], [0.025, 0.975]))
print("95% CI sigma:", np.quantile(boot_fits[:, 1], [0.025, 0.975]))

The oracle column is available only because we created the data ourselves. In practice we would have only the bootstrap column.

## The summary choice still matters

Change only the matched quantile, from `s_.75(Y) = (Ybar, q_.75(Y))` to `s_.55(Y) = (Ybar, q_.55(Y))`, and for each choice repeat the same procedure: simulate the summaries at candidate parameter values, minimize the SMM objective, repeat the entire fit across many datasets.

In [ ]:
p_pair = [0.75, 0.55]
rng_q = np.random.default_rng(51)
fits_by_p = {}
for p in p_pair:
    mhat_p = make_mhat(p)
    fits_by_p[p] = np.array([
        fit_smm_full(rng_q.normal(mu0, sigma0, size=n), p, mhat_p, W)
        for _ in range(B_outer)
    ])
    print(f"p = {p}: SD(sigma_hat) = {fits_by_p[p][:, 1].std(ddof=1):.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
for ax, p in zip(axes, p_pair):
    ax.hist(fits_by_p[p][:, 1], bins=50, color="steelblue", edgecolor="white")
    ax.axvline(sigma0, color="firebrick", linestyle="--")
    ax.set_title(f"p = {p:.2f}")
axes[-1].set_xlabel(r"$\hat\sigma$")
fig.suptitle("Sampling distribution of sigma-hat, by matched quantile")
plt.tight_layout()
plt.show()

Both summary pairs are identifiable, but the 55th percentile changes very little as $\sigma$ changes, relative to its sampling variability: the objective is much flatter in the $\sigma$ direction, estimates of $\sigma$ vary much more across datasets, optimization becomes less stable, and constrained estimates may accumulate at the boundary of the parameter space (here, the L-BFGS-B lower bound $\sigma\geq 0.05$).

# What we saw

Repeated simulation and fitting at the known truth produced the oracle sampling distribution; repeated simulation and fitting at the fitted value produced the parametric bootstrap distribution; comparing the two showed what the bootstrap is trying to approximate. Changing the summaries changed the sampling distribution, even though the generative model and sample size stayed the same.